# Tutorial 5: Train NicheTrans on STARmap PLUS data

In [6]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [7]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, use_moe_ffn=True, num_experts=1, moe_gate_hidden_dim=0, moe_gate_type='softmax', ffn_mult=2, n_top_genes=2000, workers=4, AD_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [ ]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,num_experts=args.num_experts,moe_gate_hidden_dim=args.moe_gate_hidden_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...
The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [9]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [10]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 81/81	 Loss 0.450717 (0.525637)
==> Epoch 2/20
Batch 81/81	 Loss 0.345992 (0.401744)
==> Epoch 3/20
Batch 81/81	 Loss 0.292415 (0.322680)
==> Epoch 4/20
Batch 81/81	 Loss 0.225262 (0.269787)
==> Epoch 5/20
Batch 81/81	 Loss 0.218907 (0.232797)
==> Epoch 6/20
Batch 81/81	 Loss 0.193502 (0.205857)
==> Epoch 7/20
Batch 81/81	 Loss 0.202879 (0.183796)
==> Epoch 8/20
Batch 81/81	 Loss 0.187898 (0.168966)
==> Epoch 9/20
Batch 81/81	 Loss 0.138385 (0.156508)
==> Epoch 10/20
Batch 81/81	 Loss 0.124673 (0.147808)
==> Epoch 11/20
Batch 81/81	 Loss 0.153816 (0.141639)
==> Epoch 12/20
Batch 81/81	 Loss 0.138119 (0.134316)
==> Epoch 13/20
Batch 81/81	 Loss 0.128243 (0.127428)
==> Epoch 14/20
Batch 81/81	 Loss 0.094172 (0.121326)
==> Epoch 15/20
Batch 81/81	 Loss 0.096665 (0.119480)
==> Epoch 16/20
Batch 81/81	 Loss 0.143640 (0.115098)
==> Epoch 17/20
Batch 81/81	 Loss 0.109048 (0.113968)
==> Epoch 18/20
Batch 81/81	 Loss 0.125993 (0.109032)
==> Epoch 19/20
Batch 81/81	 Loss 0.0

### Optional MoE Routing Analysis

Run the next cell after training if you want to inspect expert activation, load balance, and spatial specialization.


In [ ]:
from utils.moe_analysis import (
    analyze_moe_routing,
    plot_center_spot_activation_bar,
    plot_expert_spatial_heatmap,
    plot_slice_activation_heatmap,
    save_moe_analysis_tables,
)

moe_results = analyze_moe_routing(
    model=model,
    dataloader=testloader,
    device=device,
    include_images=False,
    include_cell_information=False,
    include_predictions=False,
    include_targets=True,
)

activation_frame = moe_results["activation_frame"]

print("Overall MoE metrics:")
print(moe_results["overall"])
display(moe_results["expert_summary"])
if not moe_results["slice_summary"].empty:
    display(moe_results["slice_summary"])
if not moe_results["region_summary"].empty:
    display(moe_results["region_summary"])

display(activation_frame.head())
plot_center_spot_activation_bar(activation_frame, row_index=0);

if activation_frame["x"].notna().any() and activation_frame["y"].notna().any():
    first_slice = activation_frame["slice_id"].dropna().iloc[0]
    plot_slice_activation_heatmap(activation_frame, slice_id=first_slice);
    plot_expert_spatial_heatmap(activation_frame, expert_index=0, slice_id=first_slice);
else:
    print(
        "Spatial coordinates were not recovered from the sample ids. "
        "Pass `sample_metadata_resolver` to `analyze_moe_routing(...)` if you want spatial heatmaps."
    )

# Optional: save the analysis tables to disk.
# save_moe_analysis_tables(moe_results, output_dir="./moe_analysis")

